# Adaptive Frequency Learning Block (AFLB)

**EvoIR Stage 1 — Core Frequency Modulation Module**

This notebook implements the complete AFLB architecture from the EvoIR paper:

1. **FD (Frequency Decomposition)** — Learnable low-pass filter to separate features into low-freq and high-freq
2. **FMgM (Frequency Modulation – generating Module)** — `pre_att`
   - High-freq branch: depthwise separable spatial conv (no FFT)
   - Low-freq branch: spectral gating via FFT → learned mask → IFFT
3. **FMoM (Frequency Modulation – operating Module)** — `FreRefine`
   - SpatialGate: max+mean pooling → 7×7 conv → sigmoid
   - ChannelGate: GAP + GMP → MLP(C→C/16→C) → sigmoid
   - Bidirectional cross-frequency modulation
4. **FreModule (Full AFLB)** — FFT-based adaptive mask + Channel Cross-Attention + FreRefine + learnable residual scaling

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numbers
from einops import rearrange

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 1. Utility Modules — LayerNorm & Reshape Helpers

In [ ]:
def to_3d(x):
    """Reshape (B, C, H, W) → (B, H*W, C) for layer norm."""
    return rearrange(x, 'b c h w -> b (h w) c')

def to_4d(x, h, w):
    """Reshape (B, H*W, C) → (B, C, H, W)."""
    return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)


class BiasFree_LayerNorm(nn.Module):
    """Layer normalization without bias term."""
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight


class WithBias_LayerNorm(nn.Module):
    """Layer normalization with learnable bias."""
    def __init__(self, normalized_shape):
        super().__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias


class LayerNorm(nn.Module):
    """Spatial-aware layer norm: flatten → normalize → reshape."""
    def __init__(self, dim, LayerNorm_type='WithBias'):
        super().__init__()
        if LayerNorm_type == 'BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)

    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)


print("✓ Utility modules defined")

## 2. FD — Frequency Decomposition (Learnable Low-Pass Filter)

Decomposes input features into **low-frequency** and **high-frequency** components using a learnable convolution-based filter that acts as an adaptive low-pass filter in the spatial domain.

**Mechanism**: `AdaptiveAvgPool2d → Conv2d → BatchNorm → Softmax → unfold/mul/sum → low`, then `high = input - low`

In [ ]:
class FD(nn.Module):
    """
    Frequency Decomposition via learnable low-pass filter.
    
    Uses adaptive average pooling + learned convolution to generate
    a dynamic low-pass filter kernel. The high-frequency component
    is obtained as the residual: high = input - low.
    
    Args:
        inchannels: Number of input channels
        kernel_size: Size of the decomposition filter (default: 3)
        stride: Convolution stride (default: 1)
        group: Number of groups for filter generation (default: 8)
    """
    def __init__(self, inchannels, kernel_size=3, stride=1, group=8):
        super().__init__()
        self.stride = stride
        self.kernel_size = kernel_size
        self.group = group
        
        # Learnable scaling for low and high frequency components
        self.lamb_l = nn.Parameter(torch.zeros(inchannels), requires_grad=True)
        self.lamb_h = nn.Parameter(torch.zeros(inchannels), requires_grad=True)
        
        # Filter generation network
        self.conv = nn.Conv2d(inchannels, group * kernel_size ** 2, 
                              kernel_size=1, stride=1, bias=False)
        self.bn = nn.BatchNorm2d(group * kernel_size ** 2)
        self.act = nn.Softmax(dim=-2)
        
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')
        
        self.pad = nn.ReflectionPad2d(kernel_size // 2)
        self.ap = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        """
        Args:
            x: Input tensor (B, C, H, W)
        Returns:
            low: Low-frequency component (B, C, H, W)
            high: High-frequency component (B, C, H, W)
        """
        identity_input = x
        
        # Generate adaptive low-pass filter from global context
        low_filter = self.ap(x)  # (B, C, 1, 1)
        low_filter = self.conv(low_filter)  # (B, group*k^2, 1, 1)
        low_filter = self.bn(low_filter)
        
        n, c, h, w = x.shape
        
        # Unfold input into patches for filtering
        x = F.unfold(
            self.pad(x), kernel_size=self.kernel_size
        ).reshape(n, self.group, c // self.group, self.kernel_size ** 2, h * w)
        
        # Reshape filter and normalize with softmax
        n, c1, p, q = low_filter.shape
        low_filter = low_filter.reshape(
            n, c1 // self.kernel_size ** 2, self.kernel_size ** 2, p * q
        ).unsqueeze(2)
        low_filter = self.act(low_filter)  # Softmax along filter dim
        
        # Apply filter: weighted sum over kernel positions
        low = torch.sum(x * low_filter, dim=3).reshape(n, c, h, w)
        
        # High frequency = residual
        high = identity_input - low
        
        return low, high


print("✓ FD (Frequency Decomposition) defined")

## 3. FMgM — Frequency Modulation Generating Module (`pre_att`)

Processes the decomposed frequency components:

- **High-frequency branch** (`flag_highF=True`): Depthwise separable convolution (1×k, k×1) + GELU activation. Operates purely in spatial domain — no FFT needed since high-frequency details are best captured spatially.

- **Low-frequency branch** (`flag_highF=False`): Spectral gating via FFT → concat real/imag → Conv1×1 → GELU → multiply → reconstruct complex → IFFT. Operating in frequency domain is natural for smooth, low-frequency structures.

In [ ]:
class FMgM(nn.Module):
    """
    Frequency Modulation generating Module (pre_att from EvoIR).
    
    Two modes:
    - High-freq (flag_highF=True): depthwise separable spatial conv
    - Low-freq (flag_highF=False): FFT → spectral gating → IFFT
    
    Args:
        in_channels: Number of input channels
        flag_highF: If True, use spatial processing; if False, use spectral
    """
    def __init__(self, in_channels, flag_highF):
        super().__init__()
        self.flag_highF = flag_highF
        k_size = 3
        dim = in_channels
        
        if flag_highF:
            # High-freq: spatial depthwise separable conv
            self.body = nn.Sequential(
                nn.Conv2d(dim, dim, (1, k_size), padding=(0, k_size // 2), groups=dim),
                nn.Conv2d(dim, dim, (k_size, 1), padding=(k_size // 2, 0), groups=dim),
                nn.GELU()
            )
        else:
            # Low-freq: spectral gating (FFT domain)
            self.body = nn.Sequential(
                nn.Conv2d(2 * dim, 2 * dim, kernel_size=1, stride=1),
                nn.GELU(),
            )

    def forward(self, ffm):
        """
        Args:
            ffm: Frequency component tensor (B, C, H, W)
        Returns:
            Modulated frequency features (B, C, H, W)
        """
        if self.flag_highF:
            # Spatial masking: conv output * input (attention-like)
            y_att = self.body(ffm) * ffm
        else:
            # Spectral gating: operate in frequency domain
            bs, c, H, W = ffm.shape
            y = torch.fft.rfft2(ffm.to(torch.float32))  # (B, C, H, W//2+1) complex
            y_imag = y.imag
            y_real = y.real
            
            # Concat real and imaginary parts
            y_f = torch.cat([y_real, y_imag], dim=1)  # (B, 2C, H, W//2+1)
            
            # Learn spectral mask via 1×1 conv
            y_att = self.body(y_f)
            y_f = y_f * y_att  # Apply learned spectral gate
            
            # Reconstruct complex tensor and IFFT back
            y_real, y_imag = torch.chunk(y_f, 2, dim=1)
            y = torch.complex(y_real, y_imag)
            y_att = torch.fft.irfft2(y, s=(H, W))  # Back to spatial domain
        
        return y_att


print("✓ FMgM (Frequency Modulation generating Module) defined")

## 4. Spatial Gate & Channel Gate (for FMoM)

These are the two gating mechanisms used in FMoM (FreRefine) for **bidirectional cross-frequency modulation**:

- **SpatialGate** (H→L Unit): Modulates low-freq with spatial attention from high-freq
- **ChannelGate** (L→H Unit): Modulates high-freq with channel attention from low-freq

In [ ]:
class SpatialGate(nn.Module):
    """
    H→L Unit: Generates spatial attention from high-frequency features.
    Uses max+mean pooling along channels → 7×7 conv → sigmoid.
    The spatial attention map highlights WHERE to enhance in low-freq.
    """
    def __init__(self):
        super().__init__()
        self.spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        max_pool = torch.max(x, 1, keepdim=True)[0]  # (B, 1, H, W)
        mean_pool = torch.mean(x, 1, keepdim=True)    # (B, 1, H, W)
        scale = torch.cat([max_pool, mean_pool], dim=1)  # (B, 2, H, W)
        scale = self.spatial(scale)  # (B, 1, H, W)
        scale = torch.sigmoid(scale)
        return scale


class ChannelGate(nn.Module):
    """
    L→H Unit: Generates channel attention from low-frequency features.
    Uses GAP + GMP → shared MLP(C→C/16→C) → sigmoid.
    The channel attention map highlights WHICH channels to enhance in high-freq.
    """
    def __init__(self, dim):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d((1, 1))
        self.max = nn.AdaptiveMaxPool2d((1, 1))
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, dim // 16, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(dim // 16, dim, 1, bias=False)
        )

    def forward(self, x):
        avg = self.mlp(self.avg(x))  # (B, C, 1, 1)
        max_pool = self.mlp(self.max(x))  # (B, C, 1, 1)
        scale = avg + max_pool
        scale = torch.sigmoid(scale)
        return scale


print("✓ SpatialGate and ChannelGate defined")

## 5. FMoM — Frequency Modulation Operating Module (`FreRefine`)

Performs **bidirectional cross-frequency modulation**:
- High-freq is modulated by **channel weights** derived from low-freq (ChannelGate)
- Low-freq is modulated by **spatial weights** derived from high-freq (SpatialGate)
- Final fusion: `out = (modulated_low + modulated_high) → 1×1 Conv`

In [ ]:
class FreRefine(nn.Module):
    """
    FMoM: Frequency Modulation operating Module.
    
    Bidirectional cross-frequency modulation:
    - low_freq *= spatial_attention(high_freq)   [WHERE to enhance]
    - high_freq *= channel_attention(low_freq)   [WHICH channels]
    - fused = projection(low + high)
    
    Args:
        dim: Number of channels
    """
    def __init__(self, dim):
        super().__init__()
        self.SpatialGate = SpatialGate()
        self.ChannelGate = ChannelGate(dim)
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, low, high):
        """
        Args:
            low:  Low-frequency features (B, C, H, W)
            high: High-frequency features (B, C, H, W)
        Returns:
            Fused and refined features (B, C, H, W)
        """
        spatial_weight = self.SpatialGate(high)     # (B, 1, H, W)
        channel_weight = self.ChannelGate(low)       # (B, C, 1, 1)
        
        high = high * channel_weight  # Modulate high-freq channels
        low = low * spatial_weight    # Modulate low-freq spatially
        
        out = low + high
        out = self.proj(out)
        return out


print("✓ FreRefine (FMoM) defined")

## 6. Channel-Wise Cross Attention

Used within the full AFLB to fuse frequency-decomposed features with backbone features via cross-attention in the channel dimension (transposed attention).

In [ ]:
class ChannelCrossAttention(nn.Module):
    """
    Channel-wise cross attention (CA).
    
    Q comes from one source (x), K/V from another (y).
    Attention is computed in channel dimension (transposed).
    
    Args:
        dim: Feature dimension
        num_head: Number of attention heads
        bias: Whether to use bias in projections
    """
    def __init__(self, dim, num_head=1, bias=False):
        super().__init__()
        self.num_head = num_head
        self.temperature = nn.Parameter(torch.ones(num_head, 1, 1), requires_grad=True)

        self.q = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)
        self.q_dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=bias)

        self.kv = nn.Conv2d(dim, dim * 2, kernel_size=1, bias=bias)
        self.kv_dwconv = nn.Conv2d(dim * 2, dim * 2, kernel_size=3, stride=1, padding=1, groups=dim * 2, bias=bias)

        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x, y):
        """
        Args:
            x: Query source (B, C, H, W)
            y: Key/Value source (B, C, H, W)
        Returns:
            Cross-attended features (B, C, H, W)
        """
        assert x.shape == y.shape, 'Shape mismatch between query and key/value sources'
        b, c, h, w = x.shape

        q = self.q_dwconv(self.q(x))
        kv = self.kv_dwconv(self.kv(y))
        k, v = kv.chunk(2, dim=1)

        # Reshape for multi-head transposed attention
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_head)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_head)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_head)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        # Channel-wise attention (transpose spatial and channel dims)
        attn = q @ k.transpose(-2, -1) * self.temperature
        attn = attn.softmax(dim=-1)

        out = attn @ v
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_head, h=h, w=w)
        out = self.project_out(out)
        return out


print("✓ ChannelCrossAttention defined")

## 7. Frequency-Guided Attention Module

Combines high-freq and low-freq guided cross-attention with adaptive fusion. Used in RES-FFTB blocks.

In [ ]:
class FrequencyGuidedAttention(nn.Module):
    """
    Cross-attention where Q comes from frequency features and K,V from backbone.
    
    Args:
        dim: Feature dimension
        num_heads: Number of attention heads
        bias: Whether to use bias
    """
    def __init__(self, dim, num_heads=4, bias=False):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        self.q_proj = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)
        self.kv_proj = nn.Conv2d(dim, dim * 2, kernel_size=1, bias=bias)
        self.q_dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=bias)
        self.kv_dwconv = nn.Conv2d(dim * 2, dim * 2, kernel_size=3, padding=1, groups=dim * 2, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x, freq_feat):
        """
        Args:
            x: Backbone features (B, C, H, W) — provides K, V
            freq_feat: Frequency features (B, C, H, W) — provides Q
        Returns:
            Frequency-guided attended features (B, C, H, W)
        """
        B, C, H, W = x.shape

        q = self.q_dwconv(self.q_proj(freq_feat))
        kv = self.kv_dwconv(self.kv_proj(x))
        k, v = kv.chunk(2, dim=1)

        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)

        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=H, w=W)
        out = self.project_out(out)
        return out


class FrequencyGuidedAttentionModule(nn.Module):
    """
    Combines high-freq and low-freq guided attention with fusion.
    
    In encoder mode: concatenate + 1×1 conv projection
    In decoder mode: adaptive prompt fusion (learnable weighted sum)
    
    Args:
        in_channels: Input channel count
        out_channels: Output channel count
        decoder_flag: Use adaptive fusion (True) or concat+conv (False)
    """
    def __init__(self, in_channels, out_channels, decoder_flag=False):
        super().__init__()
        self.decoder_flag = decoder_flag
        self.high_freq_attention = FrequencyGuidedAttention(in_channels, out_channels)
        self.low_freq_attention = FrequencyGuidedAttention(in_channels, out_channels)
        
        if self.decoder_flag:
            # Adaptive fusion with learnable weights
            self.alpha = nn.Parameter(torch.tensor(0.5))
            self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.final_proj = nn.Conv2d(in_channels * 2, out_channels, kernel_size=1)

    def forward(self, x, high_freq, low_freq):
        """
        Args:
            x: Backbone features (B, C, H, W)
            high_freq: High-frequency features (B, C, H, W)
            low_freq: Low-frequency features (B, C, H, W)
        Returns:
            Fused frequency-guided features (B, C, H, W)
        """
        high_freq_output = self.high_freq_attention(x, high_freq)
        low_freq_output = self.low_freq_attention(x, low_freq)

        if self.decoder_flag:
            alpha = torch.sigmoid(self.alpha)
            output = self.proj(alpha * high_freq_output + (1 - alpha) * low_freq_output)
        else:
            output = torch.cat([high_freq_output, low_freq_output], dim=1)
            output = self.final_proj(output)
        
        return output


print("✓ FrequencyGuidedAttentionModule defined")

## 8. Full AFLB — FreModule

The complete Adaptive Frequency Learning Block that integrates all components:

1. Input image → Conv → FFT-based adaptive frequency mask → separate high/low
2. Channel Cross-Attention between freq components and backbone features 
3. FreRefine (FMoM) bidirectional modulation
4. Final cross-attention aggregation with learnable residual scaling

In [ ]:
class AFLB(nn.Module):
    """
    Adaptive Frequency Learning Block (FreModule from EvoIR).
    
    Complete pipeline:
    1. FFT-based frequency decomposition with learnable adaptive mask
    2. Channel cross-attention between freq components and encoder features
    3. FreRefine (FMoM) bidirectional cross-frequency modulation
    4. Aggregation cross-attention + learnable residual
    
    Args:
        dim: Feature dimension (channels)
        num_heads: Number of attention heads
        bias: Use bias in projections
        in_dim: Input image channels (default: 3 for RGB)
    """
    def __init__(self, dim, num_heads=1, bias=False, in_dim=3):
        super().__init__()
        
        # Input projection
        self.conv = nn.Conv2d(in_dim, dim, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv1 = nn.Conv2d(in_dim, dim, kernel_size=3, stride=1, padding=1, bias=False)
        
        # Adaptive threshold for frequency mask
        self.score_gen = nn.Conv2d(2, 2, 7, padding=3)
        
        # Learnable residual scaling
        self.para1 = nn.Parameter(torch.zeros(dim, 1, 1))   # Scale for freq branch
        self.para2 = nn.Parameter(torch.ones(dim, 1, 1))    # Scale for identity
        
        # Cross-attention modules
        self.channel_cross_l = ChannelCrossAttention(dim, num_head=num_heads, bias=bias)
        self.channel_cross_h = ChannelCrossAttention(dim, num_head=num_heads, bias=bias)
        self.channel_cross_agg = ChannelCrossAttention(dim, num_head=num_heads, bias=bias)
        
        # Frequency refinement (FMoM)
        self.frequency_refine = FreRefine(dim)
        
        # Learnable rate for adaptive mask threshold
        self.rate_conv = nn.Sequential(
            nn.Conv2d(dim, dim // 8, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(dim // 8, 2, 1, bias=False),
        )

    def forward(self, x, y):
        """
        Args:
            x: Input image (B, in_dim, H_in, W_in)
            y: Encoder features (B, dim, H, W)
        Returns:
            Modulated features (B, dim, H, W) with learnable residual
        """
        _, _, H, W = y.size()
        x = F.interpolate(x, (H, W), mode='bilinear', align_corners=False)
        
        # FFT-based frequency decomposition with adaptive mask
        high_feature, low_feature = self.fft(x)
        
        # Cross-attention: freq components attend to encoder features
        high_feature = self.channel_cross_l(high_feature, y)
        low_feature = self.channel_cross_h(low_feature, y)
        
        # Bidirectional cross-frequency modulation (FMoM)
        agg = self.frequency_refine(low_feature, high_feature)
        
        # Final aggregation: encoder features attend to fused freq
        out = self.channel_cross_agg(y, agg)
        
        # Learnable residual connection
        return out * self.para1 + y * self.para2

    def shift(self, x):
        """Shift FFT to center (like fftshift)."""
        b, c, h, w = x.shape
        return torch.roll(x, shifts=(int(h / 2), int(w / 2)), dims=(2, 3))

    def unshift(self, x):
        """Inverse of shift (like ifftshift)."""
        b, c, h, w = x.shape
        return torch.roll(x, shifts=(-int(h / 2), -int(w / 2)), dims=(2, 3))

    def fft(self, x, n=128):
        """
        FFT-based frequency decomposition with learnable adaptive mask.
        
        1. Project input to feature space
        2. Compute adaptive threshold from global avg pool
        3. Create binary mask in frequency domain (center = low-freq)
        4. Apply mask → IFFT to get low-freq
        5. Apply (1-mask) → IFFT to get high-freq
        
        Args:
            x: Input (B, in_dim, H, W)
            n: Base divisor for threshold computation
        Returns:
            high: High-frequency features (B, dim, H, W)
            low: Low-frequency features (B, dim, H, W)
        """
        x = self.conv1(x)  # Project to dim channels
        mask = torch.zeros(x.shape, device=x.device)
        h, w = x.shape[-2:]
        
        # Adaptive threshold from global context
        threshold = F.adaptive_avg_pool2d(x, 1)  # (B, dim, 1, 1)
        threshold = self.rate_conv(threshold).sigmoid()  # (B, 2, 1, 1)
        
        # Build frequency mask per sample
        for i in range(mask.shape[0]):
            h_ = (h // n * threshold[i, 0, :, :]).int()
            w_ = (w // n * threshold[i, 1, :, :]).int()
            mask[i, :, h // 2 - h_:h // 2 + h_, w // 2 - w_:w // 2 + w_] = 1
        
        # Forward FFT
        fft = torch.fft.fft2(x, norm='forward', dim=(-2, -1))
        fft = self.shift(fft)  # Center low frequencies
        
        # High frequency: everything outside the mask
        fft_high = fft * (1 - mask)
        high = self.unshift(fft_high)
        high = torch.fft.ifft2(high, norm='forward', dim=(-2, -1))
        high = torch.abs(high)
        
        # Low frequency: inside the mask
        fft_low = fft * mask
        low = self.unshift(fft_low)
        low = torch.fft.ifft2(low, norm='forward', dim=(-2, -1))
        low = torch.abs(low)
        
        return high, low


print("✓ AFLB (Full Adaptive Frequency Learning Block) defined")

## 9. Complete FMM Pre-Work Block

The `pre_work` module combines FD decomposition → FMgM processing → FrequencyGuidedAttention.
This is the block used inside each FFTransformerBlock.

In [ ]:
class FMMPreWork(nn.Module):
    """
    Complete Frequency Modulated Module pre-work block.
    
    Pipeline:
    1. FD: Decompose input → (low_freq, high_freq)
    2. FMgM: Process each branch (spatial for high, spectral for low)
    3. FrequencyGuidedAttention: Cross-attend and fuse
    4. Residual connection with input
    
    Args:
        decoder_flag: Use adaptive fusion in decoder stages
        inchannels: Number of input channels
    """
    def __init__(self, decoder_flag=False, inchannels=48):
        super().__init__()
        self.fd = FD(inchannels)
        self.decoder_flag = decoder_flag
        
        self.freguide = FrequencyGuidedAttentionModule(
            in_channels=inchannels, 
            out_channels=inchannels, 
            decoder_flag=self.decoder_flag
        )
        
        # FMgM branches
        self.FSPG_high = FMgM(in_channels=inchannels, flag_highF=True)   # Spatial
        self.FSPG_low = FMgM(in_channels=inchannels, flag_highF=False)   # Spectral

    def forward(self, x):
        """
        Args:
            x: Input features (B, C, H, W)
        Returns:
            Frequency-modulated features (B, C, H, W) with residual
        """
        # Step 1: Decompose into frequency bands
        low_fre, high_fre = self.fd(x)
        
        # Step 2: Process each band with appropriate domain
        high_fre_aft = self.FSPG_high(high_fre)  # Spatial processing
        low_fre_aft = self.FSPG_low(low_fre)     # Spectral processing
        
        # Step 3: Cross-attend and fuse
        prompt = self.freguide(x, high_fre_aft, low_fre_aft)
        
        return prompt + x  # Residual connection


print("✓ FMMPreWork (complete pre_work block) defined")

---
## 10. Verification Tests

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Testing on: {device}")

# Test parameters
B, C, H, W = 2, 48, 64, 64
in_dim = 3

# ---- Test 1: FD (Frequency Decomposition) ----
print("\n=== Test 1: FD (Frequency Decomposition) ===")
fd = FD(C).to(device)
x_test = torch.randn(B, C, H, W, device=device)
low, high = fd(x_test)
print(f"  Input:  {x_test.shape}")
print(f"  Low:    {low.shape}")
print(f"  High:   {high.shape}")
residual = (low + high - x_test).abs().mean().item()
print(f"  Residual |low+high-input|: {residual:.6f} (should be ≈0)")
assert low.shape == x_test.shape
assert high.shape == x_test.shape
print("  ✓ PASSED")

# ---- Test 2: FMgM ----
print("\n=== Test 2: FMgM (Frequency Modulation generating Module) ===")
fmgm_high = FMgM(C, flag_highF=True).to(device)
fmgm_low = FMgM(C, flag_highF=False).to(device)
out_high = fmgm_high(high)
out_low = fmgm_low(low)
print(f"  High-freq output: {out_high.shape}")
print(f"  Low-freq output:  {out_low.shape}")
assert out_high.shape == high.shape
assert out_low.shape == low.shape
assert not torch.isnan(out_high).any(), "NaN in high-freq output!"
assert not torch.isnan(out_low).any(), "NaN in low-freq output!"
print("  ✓ PASSED")

# ---- Test 3: FreRefine (FMoM) ----
print("\n=== Test 3: FreRefine (FMoM) ===")
fmom = FreRefine(C).to(device)
fused = fmom(low, high)
print(f"  Fused output: {fused.shape}")
assert fused.shape == low.shape
assert not torch.isnan(fused).any(), "NaN in fused output!"
print("  ✓ PASSED")

# ---- Test 4: Full AFLB ----
print("\n=== Test 4: Full AFLB ===")
aflb = AFLB(dim=C, num_heads=1, bias=False, in_dim=in_dim).to(device)
x_img = torch.randn(B, in_dim, H * 2, W * 2, device=device)  # Larger input image
y_feat = torch.randn(B, C, H, W, device=device)               # Encoder features
out_aflb = aflb(x_img, y_feat)
print(f"  Image input:    {x_img.shape}")
print(f"  Encoder input:  {y_feat.shape}")
print(f"  AFLB output:    {out_aflb.shape}")
assert out_aflb.shape == y_feat.shape
assert not torch.isnan(out_aflb).any(), "NaN in AFLB output!"
print("  ✓ PASSED")

# ---- Test 5: FMMPreWork ----
print("\n=== Test 5: FMMPreWork (complete block) ===")
prework = FMMPreWork(decoder_flag=False, inchannels=C).to(device)
out_pw = prework(x_test)
print(f"  Input:  {x_test.shape}")
print(f"  Output: {out_pw.shape}")
assert out_pw.shape == x_test.shape
assert not torch.isnan(out_pw).any(), "NaN in FMMPreWork output!"
print("  ✓ PASSED")

# ---- Test 6: Gradient flow ----
print("\n=== Test 6: Gradient Flow ===")
aflb_grad = AFLB(dim=C, num_heads=1, in_dim=in_dim).to(device)
x_in = torch.randn(B, in_dim, H, W, device=device, requires_grad=True)
y_in = torch.randn(B, C, H, W, device=device, requires_grad=True)
out = aflb_grad(x_in, y_in)
loss = out.mean()
loss.backward()
print(f"  Loss: {loss.item():.6f}")
print(f"  Grad norm (x): {x_in.grad.norm().item():.6f}")
print(f"  Grad norm (y): {y_in.grad.norm().item():.6f}")
assert x_in.grad is not None, "No gradient for x!"
assert y_in.grad is not None, "No gradient for y!"
print("  ✓ PASSED")

# Parameter count
total_params = sum(p.numel() for p in aflb.parameters())
print(f"\n📊 AFLB total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print("\n✅ All AFLB tests passed!")